# Step 4 - August 2026 forecasts

**CSC3119 Assignment 4.** The brief asks for (a) a **daily** forecast for 3-16 August 2026 and (b) an **hourly** forecast for 1 August 2026, both inverse-transformed back to ug/m3. This notebook builds the future feature table, predicts, saves the CSVs, and draws the figures.

### How a future hour is described to the model

What we know for August: which sensor, where it is, its network, the hour and the month. What we do **not** know: the weather. We substitute each sensor's own **hour-of-day median** temperature and humidity (a climatology learned from every hour it ever reported). In production that column would be filled from a numerical weather forecast - the swap is one line.

### Why the daily lines are flat - and why that is honest

Nothing that varies day-to-day (weather, a festival, a fire) is known for August, and eight days of training data cannot teach weekday effects. So the model returns each sensor's *typical* August day: the level differs by place, the day-to-day wiggle is near zero. Say this in the presentation rather than pretending otherwise.

In [1]:
from pathlib import Path
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")
PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

np.random.seed(A.SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 185)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

A.set_stage("forecast")
A.banner("STEP 4 / 6  -  August 2026 forecasts")
LOG = A.PipelineLog("step4_forecasting")


  STEP 4 / 6  -  August 2026 forecasts


2026-10-18 12:34:44 | INFO | === step4_forecasting ===


## 0. Load the saved model, the grid, and the sensor registry

In [2]:
B = joblib.load(A.stage_dir("model") / "best_pm25_model.joblib")
grid = pd.read_parquet(A.stage_dir("prep") / "hourly_clean.parquet")
observed = grid[grid["is_observed"]].copy()
print(f"model      : {B['model_name']}  (sklearn {B['sklearn_version']})")
print(f"features   : {B['features']}")
print(f"memory mu/sd: {B['pm25_mean']:.3f} / {B['pm25_std']:.3f} ug/m3")
print(f"grid       : {grid.shape[0]:,} sensor-hours, {grid['device_name'].nunique()} sensors")

# Sensor registry: identity + a single position per sensor (the median fix used in
# Step 1).  This is what we feed the model for a future hour.
reg = (observed.groupby("device_name", observed=True)
       .agg(site_name=("site_name", "first"),
            network=("network", "first"),
            latitude=("latitude", "median"),
            longitude=("longitude", "median"))
       .reset_index())
print(f"registry   : {len(reg)} sensors")

# Six sensors never reported a GPS fix and no sensor sharing their site has one
# either, so they have no position.  We do not invent coordinates: the model's
# median imputer falls back to the network-centre position, and we carry a
# coordinate-known flag through every deliverable so the caveat is visible.
reg["coord_known"] = reg["latitude"].notna() & reg["longitude"].notna()
missing_pos = reg.loc[~reg["coord_known"], "device_name"].tolist()
print(f"no GPS fix : {len(missing_pos)} sensors -> {missing_pos}")
print("             (flagged; their position features are median-imputed by the model)")

model      : RandomForest  (sklearn 1.8.0)
features   : ['temp', 'humidity', 'hour_sin', 'hour_cos', 'month', 'latitude', 'longitude', 'network_airqo', 'device_name']
memory mu/sd: 32.532 / 27.862 ug/m3
grid       : 758,112 sensor-hours, 149 sensors
registry   : 149 sensors
no GPS fix : 6 sensors -> ['airqo_g5367', 'aq_43', 'aq_54', 'aq_79', 'aq_g518', 'aq_g5_50']
             (flagged; their position features are median-imputed by the model)


## 1. Weather climatology
Temperature and humidity are the only inputs we cannot observe in the future, so they are estimated per sensor per hour-of-day, with a network-wide hour-of-day fallback and then a global median. The model's missing-indicator means an imputed value is never silently trusted.

In [3]:
print("\n" + "="*78)
print(" 1. WEATHER CLIMATOLOGY (temperature / humidity are unknown in August)")
print("="*78)

# For a future hour we know the sensor, its location, the network, the hour and the
# month - but NOT the weather.  The honest substitute is each sensor's own
# hour-of-day median, learned from every observed hour it ever reported.  In
# production this column would be replaced by an actual weather forecast.
climo_dev = (observed.dropna(subset=["temp", "humidity"])
             .groupby(["device_name", "hour"], observed=True)[["temp", "humidity"]]
             .median().rename(columns={"temp": "temp_clim", "humidity": "rh_clim"}))
climo_hour = (observed.dropna(subset=["temp", "humidity"])
              .groupby("hour", observed=True)[["temp", "humidity"]]
              .median().rename(columns={"temp": "temp_clim", "humidity": "rh_clim"}))
fallback = observed[["temp", "humidity"]].median()
print(f"device x hour climatology rows : {len(climo_dev)}")
print(f"global fallback  temp={fallback['temp']:.1f}C  rh={fallback['humidity']:.1f}%")
print("A sensor with no weather history borrows the network-wide hour-of-day median;")
print("a (rare) empty hour falls back to the global median.  The model also carries a")
print("missing-indicator, so an imputed weather value is never silently trusted.")


 1. WEATHER CLIMATOLOGY (temperature / humidity are unknown in August)


device x hour climatology rows : 2875
global fallback  temp=23.7C  rh=81.1%
A sensor with no weather history borrows the network-wide hour-of-day median;
a (rare) empty hour falls back to the global median.  The model also carries a
missing-indicator, so an imputed weather value is never silently trusted.


## 2. Build the August feature table and predict

In [4]:
print("\n" + "="*78)
print(" 2. BUILD THE AUGUST FEATURE TABLE AND PREDICT")
print("="*78)

hours = pd.date_range(A.FORECAST_HOURLY, periods=16 * 24, freq="h")   # 1-16 Aug, hourly
print(f"forecast horizon: {hours[0]} .. {hours[-1]}  ({len(hours)} hours x {len(reg)} sensors)")

aug = reg.merge(pd.DataFrame({"datetime": hours}), how="cross")
aug["hour"] = aug["datetime"].dt.hour
aug = aug.merge(climo_dev.reset_index(), on=["device_name", "hour"], how="left")
aug = aug.merge(climo_hour.reset_index(), on="hour", how="left",
                suffixes=("", "_hour"))
aug["temp_clim"] = aug["temp_clim"].fillna(aug["temp_clim_hour"]).fillna(fallback["temp"])
aug["rh_clim"] = aug["rh_clim"].fillna(aug["rh_clim_hour"]).fillna(fallback["humidity"])
aug["temp"] = aug["temp_clim"]; aug["humidity"] = aug["rh_clim"]
aug = A.add_calendar_features(aug, "datetime")
aug["month"] = 8.0
print(f"feature table: {aug.shape[0]:,} rows x {len(B['features'])} model inputs")

# Predict.  The pipeline carries its own imputer/encoder, so we hand it raw features.
aug["pm25_z"] = B["model"].predict(aug[B["features"]])
aug["pm25"] = np.clip(A.inverse_z(aug["pm25_z"].to_numpy(), {"mean": B["pm25_mean"],
                                                             "std": B["pm25_std"]}), 0, None)
aug["aqi"] = A.aqi_categorise(aug["pm25"]).astype(str)
aug["aqi_code"] = A.aqi_code(aug["pm25"]).to_numpy()
aug["date"] = aug["datetime"].dt.normalize()

worked = aug[(aug["device_name"] == "aq_66") & (aug["datetime"] == "2026-08-03 08:00")]
if len(worked):
    w = worked.iloc[0]
    print(f"\nWorked inverse transform - {w['device_name']} ({w['site_name']}) "
          f"3 Aug 08:00")
    print(f"  z = {w['pm25_z']:.3f}  ->  PM2.5 = {w['pm25_z']:.3f} x {B['pm25_std']:.2f} "
          f"+ {B['pm25_mean']:.2f} = {w['pm25']:.1f} ug/m3  ->  {w['aqi']}")


 2. BUILD THE AUGUST FEATURE TABLE AND PREDICT
forecast horizon: 2026-08-01 00:00:00 .. 2026-08-16 23:00:00  (384 hours x 149 sensors)


feature table: 57,216 rows x 9 model inputs



Worked inverse transform - aq_66 (Nakasero II, Kampala 66) 3 Aug 08:00
  z = 0.055  ->  PM2.5 = 0.055 x 27.86 + 32.53 = 34.1 ug/m3  ->  Moderate


## 3. Save the two deliverables
* **Hourly**: `forecast_hourly_2026-08-01.csv` - every sensor, 24 hours.
* **Daily**: `forecast_daily_2026-08-03_to_16.csv` - mean per sensor per day. 1-2 August are skipped so the daily window is two whole Mon-Sun-style weeks; set `A.FORECAST_DAILY_START` to 1 August for a 1-14 window.

In [5]:
print("\n" + "="*78)
print(" 3. SAVE FORECAST TABLES")
print("="*78)

cols = ["device_name", "site_name", "network", "latitude", "longitude", "coord_known",
        "datetime", "pm25_z", "pm25", "aqi", "aqi_code"]
hourly = aug[cols].sort_values(["device_name", "datetime"]).reset_index(drop=True)

# Hourly deliverable: the first full day (Saturday 1 August).
hourly_1aug = hourly[hourly["datetime"].dt.normalize() == pd.Timestamp("2026-08-01")]
hourly_1aug.to_csv(A.stage_dir("forecast") / "forecast_hourly_2026-08-01.csv", index=False)

# Daily deliverable: mean of the hourly predictions per sensor per day, 3-16 August.
# dropna=False keeps the coordinate-unknown sensors in the table.
daily = (aug[(aug["datetime"] >= A.FORECAST_DAILY_START) &
             (aug["datetime"] <= A.FORECAST_DAILY_END)]
         .groupby(["device_name", "site_name", "network", "latitude", "longitude",
                   "coord_known", "date"], observed=True, dropna=False)
         .agg(pm25_mean=("pm25", "mean"), pm25_min=("pm25", "min"),
              pm25_max=("pm25", "max"))
         .reset_index())
daily["aqi"] = A.aqi_categorise(daily["pm25_mean"]).astype(str)
daily.to_csv(A.stage_dir("forecast") / "forecast_daily_2026-08-03_to_16.csv", index=False)

full_hourly = hourly.copy()
full_hourly.to_csv(A.stage_dir("forecast") / "forecast_hourly_all_sensors.csv", index=False)

print(f"hourly (1 Aug)      : {len(hourly_1aug):,} rows -> forecast_hourly_2026-08-01.csv")
print(f"daily (3-16 Aug)    : {len(daily):,} rows -> forecast_daily_2026-08-03_to_16.csv")
print(f"full hourly table   : {len(full_hourly):,} rows -> forecast_hourly_all_sensors.csv")
print("\nsensor-days by predicted band (3-16 Aug):")
print(daily["aqi"].value_counts().reindex(A.AQI_LABELS).fillna(0).astype(int).to_string())


 3. SAVE FORECAST TABLES


hourly (1 Aug)      : 3,576 rows -> forecast_hourly_2026-08-01.csv
daily (3-16 Aug)    : 2,086 rows -> forecast_daily_2026-08-03_to_16.csv
full hourly table   : 57,216 rows -> forecast_hourly_all_sensors.csv

sensor-days by predicted band (3-16 Aug):
aqi
Good                                14
Moderate                          1862
Unhealthy for Sensitive Groups     154
Unhealthy                           42
Very Unhealthy                      14
Hazardous                            0


## 4. Five showcase sensors, one per zone
149 lines on one plot is noise. We K-means the sensor coordinates into five zones and keep the sensor nearest each centre, so the figure represents the whole network rather than a handful of favourites.

In [6]:
print("\n" + "="*78)
print(" 4. SHOWCASE SENSORS - one per geographic zone (K-means)")
print("="*78)

# A map is unreadable with 149 sensors, so we summarise with five, chosen so the
# whole country is represented: K-means on the sensor coordinates, then the sensor
# nearest each cluster centre.  This is data-driven, not hand-picked.
from sklearn.cluster import KMeans
coords = reg.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
km = KMeans(n_clusters=5, n_init=10, random_state=A.SEED).fit(coords[["latitude", "longitude"]])
coords["zone"] = km.labels_
showcase = []
for z in range(5):
    grp = coords[coords["zone"] == z]
    centre = km.cluster_centers_[z]
    d = ((grp[["latitude", "longitude"]] - centre) ** 2).sum(axis=1)
    showcase.append(grp.loc[d.idxmin()])
showcase = pd.DataFrame(showcase).reset_index(drop=True)
showcase.to_csv(A.stage_dir("forecast") / "showcase_sensors.csv", index=False)
print(showcase[["device_name", "site_name", "network", "latitude", "longitude"]]
      .to_string(index=False))
showcase_names = list(showcase["device_name"])


 4. SHOWCASE SENSORS - one per geographic zone (K-means)


device_name               site_name network  latitude  longitude
      aq_66 Nakasero II, Kampala 66   airqo  0.322310  32.575754
airqo_g5477                Katukuru   airqo -0.670721  30.572341
   aq_g5_93         Gulu university   airqo  2.786563  32.309201
    aq_g530     Mpanga, Fort Portal   airqo  0.656900  30.278960
   aq_g4_95             YMCA, Jinja   airqo  0.444640  33.201200


## 5. Daily forecast figure (3-16 August)

In [7]:
print("\n" + "="*78)
print(" 5. DAILY FORECAST FIGURE (3-16 August 2026)")
print("="*78)

fig, ax = plt.subplots(figsize=(13, 5))
show = daily[daily["device_name"].isin(showcase_names)].copy()
ymax = float(show["pm25_max"].max()) * 1.08
A.shade_aqi_bands(ax, ymax)
for dev, g in show.groupby("device_name", observed=True):
    g = g.sort_values("date")
    label = f"{g['site_name'].iloc[0]} ({dev})"
    ax.plot(g["date"], g["pm25_mean"], marker="o", ms=3, lw=1.4, label=label)
ax.axhline(A.AQI_EDGES[2], color="k", ls=":", lw=0.8)
ax.text(show["date"].min(), A.AQI_EDGES[2], " USG threshold", va="bottom", fontsize=7)
ax.set_ylim(0, ymax)
ax.set_ylabel("predicted daily-mean PM2.5 (ug/m3)")
ax.set_title("Fig 4.1  Forecast daily-mean PM2.5, 3-16 August 2026 (one sensor per zone)")
ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
A.savefig(fig, "fig4_1_daily_forecast.png")
print("The lines are near-flat within a sensor: no day-to-day input (weather, event)")
print("is known for August, so the model returns each sensor's typical August day.")
print("The useful information is the LEVEL difference between sensors, not the wiggle.")
print("Date labels are placed by matplotlib; the shape of each series is what matters.")


 5. DAILY FORECAST FIGURE (3-16 August 2026)


The lines are near-flat within a sensor: no day-to-day input (weather, event)
is known for August, so the model returns each sensor's typical August day.
The useful information is the LEVEL difference between sensors, not the wiggle.
Date labels are placed by matplotlib; the shape of each series is what matters.


## 6. Hourly forecast figure with an uncertainty ribbon
The Random Forest is an average of hundreds of trees. The spread between them is a cheap stand-in for uncertainty - clearly labelled as *not* a calibrated confidence interval.

In [8]:
print("\n" + "="*78)
print(" 6. HOURLY FORECAST FIGURE (1 August, with tree-spread ribbon)")
print("="*78)

city = "aq_66"
c = (aug[(aug["device_name"] == city) &
         (aug["datetime"].dt.normalize() == pd.Timestamp("2026-08-01"))]
     .sort_values("datetime").reset_index(drop=True))

# Spread of the Random Forest's individual trees: transform once, ask every tree.
pre, clf = B["model"].named_steps["pre"], B["model"].named_steps["clf"]
Xt = pre.transform(c[B["features"]])
if hasattr(clf, "estimators_"):                     # RandomForest / bagged trees
    tree_pred = np.vstack([t.predict(Xt) for t in clf.estimators_])
    zlo = np.percentile(tree_pred, 10, axis=0)
    zhi = np.percentile(tree_pred, 90, axis=0)
else:                                               # other model: no tree spread
    zlo = zhi = c["pm25_z"].to_numpy()
lo = np.clip(A.inverse_z(zlo, {"mean": B["pm25_mean"], "std": B["pm25_std"]}), 0, None)
hi = np.clip(A.inverse_z(zhi, {"mean": B["pm25_mean"], "std": B["pm25_std"]}), 0, None)

fig, ax = plt.subplots(figsize=(12, 4.6))
ymax = float(max(hi.max(), c["pm25"].max())) * 1.15
A.shade_aqi_bands(ax, ymax)
ax.fill_between(c["datetime"], lo, hi, color="#4c72b0", alpha=0.22,
                label="10-90% spread across the forest's trees")
ax.plot(c["datetime"], c["pm25"], color="#c44e52", lw=2, marker="o", ms=3,
        label=f"mean prediction - {city}")
ax.set_ylim(0, ymax)
ax.set_ylabel("predicted PM2.5 (ug/m3)")
ax.set_title("Fig 4.2  Hourly forecast for 1 August 2026 (Nakasero II, aq_66)")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=7)
fig.tight_layout()
A.savefig(fig, "fig4_2_hourly_forecast.png")
print("The hourly curve reproduces the double daily peak learned in the EDA. The")
print("ribbon is the disagreement between the forest's trees - a rough uncertainty")
print("signal, not a calibrated confidence interval.")


 6. HOURLY FORECAST FIGURE (1 August, with tree-spread ribbon)


The hourly curve reproduces the double daily peak learned in the EDA. The
ribbon is the disagreement between the forest's trees - a rough uncertainty
signal, not a calibrated confidence interval.


## 7. Afternoon snapshot map (1 August, 13:00)
The visual answer to *'which places should be warned?'*. A boundary shapefile can be dropped into `data/` to add administrative borders, but the assignment does not require one.

In [9]:
print("\n" + "="*78)
print(" 7. AFTERNOON SNAPSHOT MAP (1 August 2026, 13:00 Kampala)")
print("="*78)

snap = hourly_1aug[hourly_1aug["datetime"] == pd.Timestamp("2026-08-01 13:00")].copy()
k = snap[(snap["latitude"].between(*A.KAMPALA_BBOX[:2])) &
         (snap["longitude"].between(*A.KAMPALA_BBOX[2:]))]
print(f"Kampala sensors at 13:00 : {len(k)}")
print("sensors in the USG-or-worse bands (orange/red):")
alert = k[k["aqi_code"] >= 2].sort_values("pm25", ascending=False)
print(alert[["device_name", "site_name", "pm25", "aqi"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 8))
lat0, lat1, lon0, lon1 = A.KAMPALA_BBOX
ax.add_patch(plt.Rectangle((lon0, lat0), lon1 - lon0, lat1 - lat0, fill=False,
                           ec="#888888", lw=0.8, ls="--"))
others = snap[~snap["device_name"].isin(k["device_name"])]
ax.scatter(others["longitude"], others["latitude"], s=14, c="#dddddd",
           edgecolor="none", label="outside Kampala")
for code, col in zip(range(6), A.AQI_COLORS):
    sub = k[k["aqi_code"] == code]
    if len(sub):
        ax.scatter(sub["longitude"], sub["latitude"], s=40 + sub["pm25"] * 1.4,
                   c=col, edgecolor="k", linewidth=0.4, alpha=0.9,
                   label=A.AQI_LABELS[code])
for _, r in alert.head(5).iterrows():
    ax.annotate(r["device_name"], (r["longitude"], r["latitude"]),
                textcoords="offset points", xytext=(6, 4), fontsize=6)
ax.set_xlim(lon0 - 0.05, lon1 + 0.05); ax.set_ylim(lat0 - 0.05, lat1 + 0.05)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Fig 4.3  Predicted PM2.5 in Kampala at 13:00, 1 August 2026")
ax.legend(fontsize=7, loc="upper left", ncol=2, handles=A.aqi_legend_handles() +
          [plt.Line2D([], [], marker="o", ls="", color="#dddddd", label="outside Kampala")])
fig.tight_layout()
A.savefig(fig, "fig4_3_map_1300.png")
print("Marker size scales with predicted PM2.5; colour is the AQI band. Even in the")
print("normally cleanest afternoon hours, several central sensors sit in USG.")


 7. AFTERNOON SNAPSHOT MAP (1 August 2026, 13:00 Kampala)
Kampala sensors at 13:00 : 95
sensors in the USG-or-worse bands (orange/red):
device_name                 site_name    pm25                            aqi
      aq_71 Mbalwa, Kira Municipality 43.0115 Unhealthy for Sensitive Groups


Marker size scales with predicted PM2.5; colour is the AQI band. Even in the
normally cleanest afternoon hours, several central sensors sit in USG.


## 8. AQI mix and a summary of the forecast

In [10]:
print("\n" + "="*78)
print(" 8. AQI MIX ACROSS ALL SENSORS")
print("="*78)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
counts = daily["aqi"].value_counts().reindex(A.AQI_LABELS).fillna(0)
axes[0].bar(range(len(counts)), counts.to_numpy(), color=A.AQI_COLORS)
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels(["Good", "Mod", "USG", "Unh", "VUnh", "Haz"], fontsize=8)
axes[0].set_ylabel("sensor-days")
axes[0].set_title("Predicted band across all sensor-days (3-16 Aug)")
for i, v in enumerate(counts.to_numpy()):
    if v:
        axes[0].text(i, v, int(v), ha="center", va="bottom", fontsize=7)

sensor_mean = (daily.groupby(["device_name", "site_name"], observed=True)["pm25_mean"]
               .mean().sort_values())
worst = sensor_mean.tail(8)
axes[1].barh(range(len(worst)), worst.to_numpy(), color="#c44e52")
axes[1].set_yticks(range(len(worst)))
axes[1].set_yticklabels([f"{s}" for d, s in worst.index], fontsize=7)
axes[1].set_xlabel("forecast daily-mean PM2.5 (ug/m3)")
axes[1].set_title("Eight most polluted sensors")
fig.tight_layout()
A.savefig(fig, "fig4_4_aqi_summary.png")
print(f"Total sensor-days Moderate-or-worse: {int(counts[1:].sum()):,} of {int(counts.sum()):,}")


 8. AQI MIX ACROSS ALL SENSORS


Total sensor-days Moderate-or-worse: 2,072 of 2,086


In [11]:
meta = {
    "horizon_hourly": str(A.FORECAST_HOURLY),
    "horizon_daily": [str(A.FORECAST_DAILY_START.date()), str(A.FORECAST_DAILY_END.date())],
    "map_snapshot": "2026-08-01 13:00",
    "n_sensors": int(len(reg)),
    "n_hours": int(len(hours)),
    "model": B["model_name"],
    "weather_source": "per-sensor hour-of-day median climatology (production: real forecast)",
    "showcase_sensors": showcase.to_dict(orient="records"),
    "sensor_day_band_counts": {k: int(v) for k, v in counts.items()},
    "worst_sensors": [{"device_name": d, "site_name": s, "pm25_mean": float(v)}
                      for (d, s), v in worst.items()],
    "kampala_alert_sensors_1300": alert[["device_name", "site_name", "pm25", "aqi"]]
        .to_dict(orient="records"),
    "env": A.env_facts(),
}
A.save_json(meta, A.stage_dir("forecast") / "forecast_meta.json")
LOG.log(f"forecast: {len(reg)} sensors x {len(hours)} h; "
        f"{int(counts[1:].sum())} sensor-days Moderate-or-worse")
LOG.save({"stage": "forecast", "n_sensors": int(len(reg)), "n_hours": int(len(hours))})

print("\n" + "="*78)
print("  STEP 4 COMPLETE")
print("="*78)
for f in sorted(A.stage_dir("forecast").glob("*")):
    print(f"  {f.name:40s} {f.stat().st_size:>12,d} bytes")
print("\nNext:  Step5_Monitoring.ipynb (Evidently drift + model quality)")

2026-10-18 12:35:01 | INFO | forecast: 149 sensors x 384 h; 2072 sensor-days Moderate-or-worse

  STEP 4 COMPLETE
  fig4_1_daily_forecast.png                      70,706 bytes
  fig4_2_hourly_forecast.png                    103,191 bytes
  fig4_3_map_1300.png                           101,669 bytes
  fig4_4_aqi_summary.png                         72,270 bytes
  forecast_daily_2026-08-03_to_16.csv           289,885 bytes
  forecast_hourly_2026-08-01.csv                495,492 bytes
  forecast_hourly_all_sensors.csv             7,926,432 bytes
  forecast_meta.json                              3,322 bytes
  showcase_sensors.csv                              371 bytes

Next:  Step5_Monitoring.ipynb (Evidently drift + model quality)
